In [1]:
import pandas as pd
df=pd.read_csv('cleaned_data.csv')

print(df.isnull().sum())
print(df.duplicated().sum())
df.sample(10)




answers    0
label      0
dtype: int64
0


,answers,label
2154,"[""A second mortgage is basically a home equity...",0
42277,"[""Exchange-traded funds (ETFs) and mutual fund...",1
17405,['http://www.irs.gov/taxtopics/tc503.html says...,0
19481,['Massachusets does no such thing. The 5.25% t...,0
20418,"['Judgement, settlement, insurance proceeds, e...",0
37748,['A producer is a person who is responsible fo...,1
36190,"['Most cell phones have a small, long-lasting ...",1
32929,['The English language has a long and complex ...,1
44898,['It sounds like you are experiencing anxiety ...,1
30787,"[""New cars have a distinctive smell because th...",1


In [2]:
import ast

def clean_text(x):
    if isinstance(x, list):
        return " ".join(map(str, x))

    if isinstance(x, str):
        try:
            parsed = ast.literal_eval(x)

            if isinstance(parsed, list):
                return " ".join(map(str, parsed))
            else:
                return str(parsed)

        except:
            return x   # IMPORTANT: fallback safe

    return str(x)



In [3]:
df['answers'] = df['answers'].str.lower()
df['answers']=df['answers'].apply(clean_text)

df.head()

,answers,label
0,"basically there are many categories of "" best ...",0
1,salt is good for not dying in car crashes and ...,0
2,the way it works is that old tv stations got a...,0
3,you ca n't just go around assassinating the le...,0
4,wanting to kill the shit out of germans drives...,0


In [4]:
df_feat = df.copy()
df_feat.sample(5)

,answers,label
7462,"30 mm refers to the caliber , which is the dia...",0
16169,there are approximately 642 skeletal muscles w...,0
24043,planets are round because of the force of grav...,1
28458,"in the united states, recipes often use cups a...",1
45451,i'm sorry to hear about your father's diagnosi...,1


In [5]:
import spacy
nlp = spacy.load("en_core_web_sm", disable=["ner", "parser"])
nlp.add_pipe("sentencizer")

In [6]:
def load_word_list(path):
    with open(path, "r", encoding="utf-8") as f:
        return set(line.strip().lower() for line in f if line.strip())

chat_words = load_word_list("chat_words.txt")
function_words = load_word_list("function_words.txt")
discourse_markers = load_word_list("discourse_markers.txt")

In [7]:
def split_sentences(text):
    doc = nlp(text)
    return [sent.text.strip() for sent in doc.sents if sent.text.strip()]

In [8]:
def extract_features(text):
    doc = nlp(text)

    tokens = [t.text.lower() for t in doc if not t.is_space]
    alpha_tokens = [t.text.lower() for t in doc if t.is_alpha]

    total_tokens = len(tokens)
    total_alpha = len(alpha_tokens)

    if total_tokens == 0 or total_alpha == 0:
        return {
            "chat_word_count": 0,
            "punct_count": 0,
            "function_count": 0,
            "discourse_count": 0,
            "spelling_error_ratio": 0,
            "ttr": 0,
            "function_word_ratio": 0,
            "discourse_ratio": 0,
            "sentence_length": 0,
            "avg_word_length": 0,
        }

    chat_word_count = sum(1 for t in tokens if t in chat_words)
    punct_count = sum(1 for t in doc if t.is_punct)

    function_count = sum(1 for t in tokens if t in function_words)
    discourse_count = sum(1 for t in tokens if t in discourse_markers)

    spelling_errors = sum(1 for t in doc if t.is_alpha and t.is_oov)

    return {
        "chat_word_count": chat_word_count,
        "punct_count": punct_count,
        "function_count": function_count,
        "discourse_count": discourse_count,
        "spelling_error_ratio": spelling_errors / total_alpha,
        "ttr": len(set(alpha_tokens)) / total_alpha,
        "function_word_ratio": function_count / total_tokens,
        "discourse_ratio": discourse_count / total_tokens,
        "sentence_length": total_tokens,
        "avg_word_length": sum(len(t) for t in alpha_tokens) / total_alpha,
    }

In [9]:
def aggregate_sentence_features(sentences):
    feats = [extract_features(s) for s in sentences]

    if len(feats) == 0:
        return {}

    df = pd.DataFrame(feats)

    return {
        "chat_word_count": df["chat_word_count"].sum(),
        "punct_count": df["punct_count"].sum(),

        "function_count": df["function_count"].sum(),
        "discourse_count": df["discourse_count"].sum(),

        "spelling_error_ratio": df["spelling_error_ratio"].mean(),
        "ttr": df["ttr"].mean(),
        "function_word_ratio": df["function_word_ratio"].mean(),
        "discourse_ratio": df["discourse_ratio"].mean(),

        "sentence_length_mean": df["sentence_length"].mean(),
        "sentence_length_std": df["sentence_length"].std(),

        "avg_word_length": df["avg_word_length"].mean(),
    }

In [10]:
def build_final_features(df):
    rows = []

    for _, row in df.iterrows():
        text = row["answers"]
        label = row["label"]

        # 1. split into sentences
        sentences = split_sentences(text)

        # 2. aggregate sentence-level features → document-level features
        features = aggregate_sentence_features(sentences)

        # 3. attach label
        features["label"] = label

        rows.append(features)

    return pd.DataFrame(rows)

In [11]:
#df_final = build_final_features(df_feat)

In [12]:
#import joblib
#joblib.dump(df_final, "final_features.pkl")

In [13]:
import joblib
df_feat = joblib.load("final_features.pkl")

In [14]:
df_feat.head()

,chat_word_count,punct_count,function_count,discourse_count,spelling_error_ratio,ttr,function_word_ratio,discourse_ratio,sentence_length_mean,sentence_length_std,avg_word_length,label
0,2.0,48.0,118.0,15.0,1.0,0.899103,0.442874,0.059134,24.272727,14.185139,4.373566,0
1,1.0,46.0,192.0,24.0,1.0,0.918157,0.458618,0.049649,20.750000,10.289877,4.537067,0
2,4.0,55.0,196.0,24.0,1.0,0.900015,0.391254,0.042689,36.923077,23.514044,4.643522,0
3,1.0,20.0,92.0,12.0,1.0,0.933176,0.415362,0.055521,22.111111,19.303137,4.923874,0
4,1.0,29.0,147.0,15.0,1.0,0.868949,0.512040,0.050100,31.666667,12.549900,4.692924,0


In [15]:
df_final = pd.concat(
    [
        df[["answers", "label"]].reset_index(drop=True),
        df_feat.drop(columns=["label"], errors="ignore").reset_index(drop=True)
    ],
    axis=1
)
df_final.head()

,answers,label,chat_word_count,punct_count,function_count,discourse_count,spelling_error_ratio,ttr,function_word_ratio,discourse_ratio,sentence_length_mean,sentence_length_std,avg_word_length
0,"basically there are many categories of "" best ...",0,2.0,48.0,118.0,15.0,1.0,0.899103,0.442874,0.059134,24.272727,14.185139,4.373566
1,salt is good for not dying in car crashes and ...,0,1.0,46.0,192.0,24.0,1.0,0.918157,0.458618,0.049649,20.750000,10.289877,4.537067
2,the way it works is that old tv stations got a...,0,4.0,55.0,196.0,24.0,1.0,0.900015,0.391254,0.042689,36.923077,23.514044,4.643522
3,you ca n't just go around assassinating the le...,0,1.0,20.0,92.0,12.0,1.0,0.933176,0.415362,0.055521,22.111111,19.303137,4.923874
4,wanting to kill the shit out of germans drives...,0,1.0,29.0,147.0,15.0,1.0,0.868949,0.512040,0.050100,31.666667,12.549900,4.692924


In [16]:
df_feat.head()

,chat_word_count,punct_count,function_count,discourse_count,spelling_error_ratio,ttr,function_word_ratio,discourse_ratio,sentence_length_mean,sentence_length_std,avg_word_length,label
0,2.0,48.0,118.0,15.0,1.0,0.899103,0.442874,0.059134,24.272727,14.185139,4.373566,0
1,1.0,46.0,192.0,24.0,1.0,0.918157,0.458618,0.049649,20.750000,10.289877,4.537067,0
2,4.0,55.0,196.0,24.0,1.0,0.900015,0.391254,0.042689,36.923077,23.514044,4.643522,0
3,1.0,20.0,92.0,12.0,1.0,0.933176,0.415362,0.055521,22.111111,19.303137,4.923874,0
4,1.0,29.0,147.0,15.0,1.0,0.868949,0.512040,0.050100,31.666667,12.549900,4.692924,0


In [17]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from scipy.sparse import hstack

In [21]:
X_ans = df_final["answers"]
y = df_final["label"]
X_custom = df_final.drop(columns=["answers", "label"])

In [24]:
X_ans_train, X_text_test, X_custom_train, X_custom_test, y_train, y_test = train_test_split(
    X_ans,
    X_custom,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y  # Maintain class balance
)